# 日内异常成交量价格压力因子：新版因子引擎能力示例

本 notebook 使用 `SmartQuantDataProvider` 直连真实日历、资产轴和分钟数据源，通过新版 `factor_engine` 计算一个 **日内异常成交量价格压力（Abnormal Volume Price Pressure, AVPP）** 因子。

它的重点不仅是得到一个因子值，而是用一条真实量价公式集中展示当前引擎的核心能力：

```text
1min close / volume
        ↓
分钟收益 + 同 step 跨日成交量基准
        ↓
异常成交量加权的分钟价格压力 pressure_1m
        ↓
resample: 1min → 5min
        ↓
同一个 pressure_5m 分叉复用
   ┌──────┼────────┬──────────┐
   |      |        |          |
 step_sum step_std realized-vol step_kurtosis
   |      |        |          |
   └──────┴────────┴──────────┘
        ↓  intraday → 1d Domain lowering
日频统计量
        ↓
ts_mean / ts_std
        ↓
多个日频输出共享一张 Term DAG
```



## 因子定义

对股票 $i$、交易日 $d$、分钟 step $t$：

1. 分钟对数收益：

$$
r_{d,i,t}=\log\left(\frac{P_{d,i,t}}{P_{d,i,t-1}}\right)
$$

`step_delay()` 只在当日 step 轴上移动，因此每天第一根 bar 为缺失，不会把前一日收盘到当日 09:30 混成一分钟收益。

2. 过去 20 日同一分钟位置的平均成交量基准，仅使用 **当日之前** 的数据：

$$
\bar V^{20}_{d,i,t}=Mean(V_{d-20,i,t},\ldots,V_{d-1,i,t})
$$

3. 异常成交量：

$$
AV_{d,i,t}=\frac{V_{d,i,t}}{\bar V^{20}_{d,i,t}}
$$

4. 分钟价格压力：

$$
Pressure_{d,i,t}=r_{d,i,t}\times AV_{d,i,t}
$$

随后先把 1min pressure 聚合到 5min，再从同一个 `pressure_5m` 计算：

- `pressure_sum_d`：日内净方向压力；
- `pressure_std_d`：5min pressure 的日内离散度；
- `pressure_vol_d = sqrt(sum(pressure_5m²))`：日内 realized pressure magnitude；
- `pressure_kurt_d`：日内 pressure 超额峰度。

主信号先定义方向效率：

$$
Efficiency_d=\frac{pressure\_sum_d}{pressure\_vol_d}
$$

最终 AVPP：

$$
AVPP_d=\frac{Mean_{5d}(Efficiency)}{Std_{20d}(Efficiency)}
$$

它表示近期异常成交量驱动的方向压力是否持续、且相对于自身历史波动是否足够稳定。因子正负方向是否最终用于 momentum 或 reversal，应由实证决定，本 notebook 不预设 alpha 方向。


## 1. 导入新版公共 API

In [ ]:
from contextlib import contextmanager
from dataclasses import replace
from functools import wraps
from pathlib import Path
import os
import resource
import sys
import threading
from time import perf_counter

import numpy as np
import pandas as pd


def find_project_root(start=None):
    """向上查找包含新版 src/factor_engine 包的项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "factor_engine").exists():
            return path
    raise RuntimeError("找不到包含 src/factor_engine 的项目根目录")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from factor_engine import (  # noqa: E402
    BatchFactorEngine,
    ComputeRequest,
    DomainSpec,
    ExecutionOptions,
    FormulaBatch,
    SmartQuantDataProvider,
    source,
)
from factor_engine.operators import default_operator_registry  # noqa: E402

PROJECT_ROOT


## 2. 设置真实数据和计算参数

本 notebook 默认走正式的无 Store 数据链路：

- `SmartQuantDataProvider` 从正式数据源解析交易日轴和股票 master axis；
- Catalog 将 `stk.1min.close_price`、`stk.1min.turnover_volume` 解析到真实分钟 Parquet；
- Compiler 自动从公式 DAG 推导 lookback，不手填 overlap。

本公式的理论最大 lookback 是 39 个交易日：成交量同-step 20 日基准需要 20 日历史，随后 20 日 rolling 再增加 19 日。


In [ ]:
START = "2025-01-01"
END = "2025-12-31"

CHUNK_SIZE = 100
MEMORY_SAMPLE_SECONDS = 0.05
OUTPUT_CSV = Path("/tmp/intraday_abnormal_volume_price_pressure_full.csv")
PREVIEW_ROWS = 10

FORMULA_IDS = [
    "abnormal_volume_price_pressure",
    "pressure_sum_5d",
    "pressure_std_20d",
    "pressure_vol_20d",
    "pressure_kurtosis_20d",
]


## 3. 创建 DataProvider

每个任务创建一个独立的 `SmartQuantDataProvider`。它直接读取正式 Catalog、交易日历、资产轴和物理数据，不创建 Snapshot Store。数据库连接参数从环境变量或项目根目录 `.env` 读取。


In [ ]:
provider = SmartQuantDataProvider()

print("provider:", type(provider).__name__)
print("request date range:", START, "->", END)
print("catalog fingerprint:", provider.catalog_fingerprint)
print("catalog source count:", provider.catalog.source_count)


## 4. 检查正式 Catalog 中的分钟数据源

这里只通过 DataProvider 公共接口提前查看输入 Domain 描述。真正的 `SourceSpec` 绑定和数据读取仍发生在每个物理分区的 `bind_many()/load_many()` 阶段。


In [ ]:
required_keys = [
    "stk.1min.close_price",
    "stk.1min.turnover_volume",
]

source_refs = [source(key) for key in required_keys]
source_specs = provider.describe_many(source_refs)
source_df = pd.DataFrame(
    [
        {
            "key": ref.logical_key,
            "asset": spec.asset_type,
            "frequency": spec.frequency,
            "step_count": spec.step_count,
            "value_kind": spec.value_kind.value,
        }
        for ref, spec in source_specs.items()
    ]
)

source_df


## 5. 定义共享 FormulaBatch

这里刻意把核心中间指标放进 `common_inputs`，而不是在 5 个输出公式中重复书写。

这会展示两层共享：

1. **符号层共享**：`pressure_5m`、`pressure_sum_d`、`pressure_std_d`、`pressure_vol_d`、`pressure_kurt_d` 都只定义一次；
2. **Term DAG / CSE 层共享**：Compiler 对等价表达式只生成一个 Term，多输出通过 reference count 共同消费。

注意成交量基准使用 `delay(minute_volume, periods=1, axis=0)` 后再做 `intraday_by_step_mean(..., 20)`，因此当天成交量不会进入当天自己的历史基准。


In [ ]:
batch = FormulaBatch.from_text(
    common_inputs="""
        minute_close = source("stk.1min.close_price")
        minute_volume = source("stk.1min.turnover_volume")

        minute_ret = ln(
            minute_close / step_delay(minute_close, periods=1)
        )

        volume_lag1 = delay(minute_volume, periods=1, axis=0)
        volume_baseline_20 = intraday_by_step_mean(
            volume_lag1,
            window_days=20,
        )
        abnormal_volume = minute_volume / volume_baseline_20

        pressure_1m = minute_ret * abnormal_volume
        pressure_5m = resample(
            pressure_1m,
            "5min",
            method="sum",
        )

        pressure_sq_5m = pressure_5m * pressure_5m

        
        pressure_std_d = step_std(pressure_5m)
        pressure_vol_d = sqrt(step_sum(pressure_sq_5m))
        pressure_kurt_d = step_kurtosis(pressure_5m)

        pressure_efficiency_d = pressure_sum_d / pressure_vol_d
    """,
    formulas={
        "abnormal_volume_price_pressure": """
            pressure_fast = ts_mean(
                pressure_efficiency_d,
                window=5,
                min_periods=3,
            )
            pressure_risk = ts_std(
                pressure_efficiency_d,
                window=20,
                min_periods=10,
            )
            factor = pressure_fast / pressure_risk
        """,
        "pressure_sum_5d": """
            factor = ts_mean(
                pressure_sum_d,
                window=5,
                min_periods=3,
            )
        """,
        "pressure_std_20d": """
            factor = ts_mean(
                pressure_std_d,
                window=20,
                min_periods=10,
            )
        """,
        "pressure_vol_20d": """
            factor = ts_mean(
                pressure_vol_d,
                window=20,
                min_periods=10,
            )
        """,
        "pressure_kurtosis_20d": """
            factor = ts_mean(
                pressure_kurt_d,
                window=20,
                min_periods=10,
            )
        """,
    },
)


## 6. 配置逐步耗时与 RSS 内存采样

Profiler 只存在于本 notebook，不修改引擎协议：

- 包装 `compile()`、每次 `provider.load_many()` 和每个 operator 调用，记录 wall time；
- 每隔 `MEMORY_SAMPLE_SECONDS` 秒从 `/proc/self/statm` 读取当前进程 RSS；
- 采样点同时记录当时执行的 phase，计算结束后生成步骤表和内存曲线；
- 不依赖 `psutil`、`matplotlib` 等额外包。


In [ ]:
def current_rss_mib():
    """读取当前 Python 进程 RSS；非 Linux 环境退化为进程历史峰值。"""
    try:
        with open("/proc/self/statm", encoding="ascii") as statm:
            resident_pages = int(statm.read().split()[1])
        return resident_pages * os.sysconf("SC_PAGE_SIZE") / 1024**2
    except (FileNotFoundError, IndexError, ValueError):
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak / 1024**2 if sys.platform == "darwin" else peak / 1024


class ComputeProfiler:
    def __init__(self, sample_seconds=0.05):
        self.sample_seconds = sample_seconds
        self.events = []
        self.samples = []
        self._counts = {}
        self._phase = "idle"
        self._stop = threading.Event()
        self._thread = None
        self._started = None

    def _sample(self):
        self.samples.append(
            {
                "seconds": perf_counter() - self._started,
                "rss_mib": current_rss_mib(),
                "phase": self._phase,
            }
        )

    def _sample_loop(self):
        while not self._stop.wait(self.sample_seconds):
            self._sample()

    def start(self):
        self.events.clear()
        self.samples.clear()
        self._counts.clear()
        self._stop.clear()
        self._started = perf_counter()
        self._sample()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        self._thread.join()
        self._sample()

    @contextmanager
    def span(self, category, step, **details):
        key = (category, step)
        self._counts[key] = self._counts.get(key, 0) + 1
        call = self._counts[key]
        started = perf_counter()
        start_seconds = started - self._started
        rss_start = current_rss_mib()
        sample_start = len(self.samples)
        previous_phase = self._phase
        self._phase = f"{category}:{step}"
        status = "ok"
        try:
            yield
        except Exception:
            status = "error"
            raise
        finally:
            rss_end = current_rss_mib()
            sampled = [row["rss_mib"] for row in self.samples[sample_start:]]
            self.events.append(
                {
                    "category": category,
                    "step": step,
                    "call": call,
                    "status": status,
                    "start_seconds": start_seconds,
                    "elapsed_ms": (perf_counter() - started) * 1000,
                    "rss_start_mib": rss_start,
                    "rss_end_mib": rss_end,
                    "rss_peak_mib": max([rss_start, rss_end, *sampled]),
                    "rss_delta_mib": rss_end - rss_start,
                    **details,
                }
            )
            self._phase = previous_phase


def wrap_operator(profiler, name, func):
    @wraps(func)
    def measured(*args, **kwargs):
        with profiler.span("operator", name):
            return func(*args, **kwargs)

    return measured


class ProfiledBatchFactorEngine(BatchFactorEngine):
    def __init__(self, provider, profiler):
        self.profiler = profiler
        operators = {
            name: replace(spec, func=wrap_operator(profiler, name, spec.func))
            for name, spec in default_operator_registry().items()
        }
        super().__init__(provider, operators=operators)

    def compile(self, request):
        with self.profiler.span("engine", "compile"):
            return super().compile(request)


## 7. 创建请求并执行带 profiling 的计算

所有输出都是 `1d × 1`。Profiler 在 `compute()` 返回或异常退出后都会停止采样并恢复原始 `provider.load_many()`。


In [ ]:
request = ComputeRequest(
    domain=DomainSpec(
        start=START,
        end=END,
        asset_scope={"stk": "all"},
        target_asset="stk",
        target_freq="1d",
        target_step_count=1,
    ),
    batch=batch,
)

profiler = ComputeProfiler(MEMORY_SAMPLE_SECONDS)
engine = ProfiledBatchFactorEngine(provider, profiler)
provider_event_start = len(provider.diagnostics)
original_load_many = provider.load_many

def measured_load_many(bindings):
    fields = ", ".join(
        binding.source_spec.field or binding.source_spec.name
        for binding in bindings
    )
    with profiler.span("provider", "load_many", fields=fields):
        return original_load_many(bindings)

provider.load_many = measured_load_many
profiler.start()
try:
    with profiler.span("engine", "compute_total"):
        result = engine.compute(
            request,
            options=ExecutionOptions(chunk_size=CHUNK_SIZE),
        )
finally:
    profiler.stop()
    provider.load_many = original_load_many

print("semantic_id:", result.plan.semantic_id)
print("job lookback:", result.plan.job_lookback)
print("output formulas:", list(result.plan.outputs))
print("resolved output shape:", result.domain.shape)


## 8. 查看逐步耗时与内存曲线

`profile_summary_df` 按步骤聚合所有 partition 的调用；`profile_events_df` 保留每次调用明细。Provider 表进一步拆出 `calendar / asset_axis / code_map / load` 等物理查询。RSS 曲线的 tooltip 会显示采样时正在运行的 phase。

> Operator wrapper 会带来少量 Python 计时开销；50ms RSS 采样可能漏掉更短的瞬时峰值，因此结果适合定位热点，不作为精密 benchmark。


In [ ]:
profile_events_df = pd.DataFrame(profiler.events).sort_values(
    ["start_seconds", "category", "step"]
).reset_index(drop=True)
profile_summary_df = (
    profile_events_df.groupby(["category", "step"], as_index=False)
    .agg(
        calls=("elapsed_ms", "size"),
        total_ms=("elapsed_ms", "sum"),
        mean_ms=("elapsed_ms", "mean"),
        max_ms=("elapsed_ms", "max"),
        peak_rss_mib=("rss_peak_mib", "max"),
        net_rss_delta_mib=("rss_delta_mib", "sum"),
    )
    .sort_values("total_ms", ascending=False)
    .reset_index(drop=True)
)
memory_profile_df = pd.DataFrame(profiler.samples)
provider_profile_df = pd.DataFrame(
    provider.diagnostics[provider_event_start:]
)

print("profiled wall seconds:", round(memory_profile_df["seconds"].iloc[-1], 3))
print("start RSS MiB:", round(memory_profile_df["rss_mib"].iloc[0], 1))
print("peak RSS MiB:", round(memory_profile_df["rss_mib"].max(), 1))
display(profile_summary_df)

provider_columns = [
    name
    for name in [
        "operation", "dataset", "fields", "start", "end",
        "rows", "bytes", "elapsed_ms", "status",
    ]
    if name in provider_profile_df.columns
]
display(provider_profile_df[provider_columns])
display(profile_events_df)

memory_curve = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "width": 900,
    "height": 320,
    "data": {"values": memory_profile_df.to_dict("records")},
    "mark": {"type": "line", "tooltip": True},
    "encoding": {
        "x": {"field": "seconds", "type": "quantitative", "title": "Elapsed seconds"},
        "y": {
            "field": "rss_mib",
            "type": "quantitative",
            "title": "Process RSS (MiB)",
            "scale": {"zero": False},
        },
        "tooltip": [
            {"field": "seconds", "type": "quantitative", "format": ".3f"},
            {"field": "rss_mib", "type": "quantitative", "format": ".1f"},
            {"field": "phase", "type": "nominal"},
        ],
    },
}
display({"application/vnd.vegalite.v5+json": memory_curve}, raw=True)


## 9. Review LogicalPlan：Domain lowering、lookback 与 operator 组成

这个表直接展示每个 Term 的：

- operator；
- 输入 Term；
- 累计 lookback；
- frequency / step_count；
- reference count。

修复后的关键契约应当清晰出现：

```text
pressure_1m  --resample--> 5min × 48
pressure_5m --step_*-----> 1d × 1
1d statistic --ts_*------> 1d × 1
```


In [ ]:
term_rows = []
for term_id in result.plan.topological_order:
    term = result.plan.terms[term_id]
    term_rows.append(
        {
            "term_id": term_id,
            "type": type(term).__name__,
            "operator": getattr(term, "operator_name", None),
            "inputs": getattr(term, "input_term_ids", None),
            "lookback": term.lookback,
            "frequency": None if term.domain is None else term.domain.frequency,
            "steps": None if term.domain is None else term.domain.step_count,
            "reference_count": result.plan.reference_counts[term_id],
        }
    )

plan_df = pd.DataFrame(term_rows)
print("term count:", len(plan_df))
print("source term count:", (plan_df["type"] == "SourceTerm").sum())
plan_df


## 10. 验证 step reduction

这里不是只看最终 shape，而是检查 LogicalPlan 中 step reduction Term 的 **Domain frequency**。

对本公式而言，`step_sum / step_std / step_kurtosis` 的输入是 `5min × 48`，输出必须是 `1d × 1`。


In [ ]:
step_reduce_ops = {
    "step_mean",
    "step_sum",
    "step_std",
    "step_max",
    "step_min",
    "step_first",
    "step_last",
    "step_kurtosis",
    "step_corr",
}

step_reduce_df = plan_df[
    plan_df["operator"].isin(step_reduce_ops)
][[
    "term_id",
    "operator",
    "inputs",
    "lookback",
    "frequency",
    "steps",
    "reference_count",
]].reset_index(drop=True)

assert not step_reduce_df.empty
assert (step_reduce_df["frequency"] == "1d").all(), step_reduce_df
assert (step_reduce_df["steps"] == 1).all(), step_reduce_df

print("step reduction Domain contract: OK")
step_reduce_df


## 11. 直接观察 common input / CSE 共享

共享节点最直观的证据是 `reference_count > 1`。

本例中 `pressure_5m` 只应存在一个 `resample(1min → 5min, sum)` Term，但随后被平方、`step_sum`、`step_std`、`step_kurtosis` 等多条链共同消费。这样分钟计算和 1→5min 聚合不会因为多个输出公式重复执行。


In [ ]:
shared_terms_df = plan_df[
    plan_df["reference_count"] > 1
][[
    "term_id",
    "type",
    "operator",
    "inputs",
    "lookback",
    "frequency",
    "steps",
    "reference_count",
]].sort_values("reference_count", ascending=False)

resample_hubs = shared_terms_df[
    shared_terms_df["operator"] == "resample"
]

assert len(resample_hubs) == 1, resample_hubs
assert int(resample_hubs.iloc[0]["reference_count"]) >= 4

print("shared resample hub:")
display(resample_hubs)
print("all shared Terms:")
shared_terms_df


## 12. 查看自动 lookback 推导

这条公式同时含有两层历史依赖：

```text
minute_volume
  ↓ delay(axis=0, 1)               +1 day
  ↓ intraday_by_step_mean(20)      +19 days
  = pressure daily statistics      20 days cumulative
  ↓ ts_std / ts_mean(window=20)    +19 days
  = job_lookback                   39 days
```

下面按 lookback 排序，确认 Compiler 是沿真实共享 DAG 自动累计，而不是 notebook 手工指定 overlap。


In [ ]:
lookback_df = plan_df[
    plan_df["type"] != "LiteralTerm"
][[
    "term_id",
    "operator",
    "lookback",
    "frequency",
    "steps",
    "reference_count",
]].sort_values(["lookback", "operator"], ascending=[False, True])

print("job_lookback:", result.plan.job_lookback)
lookback_df.head(15)


## 13. 使用引擎装配的全部输出

一个 `FormulaBatch` 有 5 个 output，但它们共享同一 LogicalPlan 和同一批 SourceTerm。

`engine.compute()` 已在内部消费 `ResultStream`，并把每个 `formula_id` 装配成完整的 `T × N × 1` 数组。这里直接把 `result.arrays` 转成便于分析的 DataFrame，并统计逐日覆盖率。


In [ ]:
result_dfs = {
    formula_id: pd.DataFrame(
        values[:, :, 0],
        index=pd.Index(result.domain.dates.astype(str), name="DataDate"),
        columns=pd.Index(result.domain.codes, name="InnerCode"),
    )
    for formula_id, values in result.arrays.items()
}

coverage_df = pd.concat(
    [
        pd.DataFrame(
            {
                "formula_id": formula_id,
                "DataDate": frame.index,
                "coverage": frame.notna().mean(axis=1),
                "finite_count": frame.notna().sum(axis=1),
            }
        )
        for formula_id, frame in result_dfs.items()
    ],
    ignore_index=True,
)

print("load calls:", result.stats.load_calls)
print("peak workspace values:", result.stats.peak_workspace_values)
print("released terms:", len(result.stats.released_terms))
print("provider events:", len(result.stats.provider_events))
print("result shape per formula:", result.domain.shape)
coverage_df.groupby("formula_id", as_index=False).tail(1)


## 14. 查看主因子完整宽表与覆盖率

`result_dfs["abnormal_volume_price_pressure"]` 是请求区间内完整 `date × InnerCode` 日频结果。先查看主因子，再看最近日期的覆盖率。


In [ ]:
factor_df = result_dfs["abnormal_volume_price_pressure"]
display(factor_df)

main_coverage = coverage_df[
    coverage_df["formula_id"] == "abnormal_volume_price_pressure"
].reset_index(drop=True)
main_coverage.tail(PREVIEW_ROWS)


## 15. 同时查看五个共享输出的最新截面

这一页能直观看到同一个 `pressure_5m` 被分叉成不同日内统计，再经过不同日频 rolling 后形成多个结果：

- AVPP 主因子；
- 净压力 5 日均值；
- 日内 pressure 标准差 20 日均值；
- realized pressure magnitude 20 日均值；
- 日内 pressure 峰度 20 日均值。


In [ ]:
target_date = str(result.domain.dates[-1])
latest_output_df = pd.DataFrame(
    {
        formula_id: result_dfs[formula_id].iloc[-1]
        for formula_id in FORMULA_IDS
    }
)
latest_output_df.index.name = "InnerCode"

print("target_date:", target_date)
display(latest_output_df)
latest_output_df.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


## 16. 检查主因子与诊断统计的关系

这里不做 alpha 有效性结论，只做工程/研究 sanity check：查看最新截面上主因子和几个底层统计之间的 Pearson 相关性，确认不同分支确实产生了不同信息，而不是意外重复同一个输出。


In [ ]:
latest_output_df.corr()


## 17. 生成完整长表并导出

把 5 个输出合并成同一份 `runner_date × InnerCode` 长表。这样既可以 review 主因子，也可以保留诊断字段用于后续研究或与独立实现做数值核对。


In [ ]:
n_dates = len(result.domain.dates)
n_codes = len(result.domain.codes)

full_result_df = pd.DataFrame(
    {
        "runner_date": np.repeat(result.domain.dates.astype(str), n_codes),
        "InnerCode": np.tile(result.domain.codes, n_dates),
    }
)

for formula_id in FORMULA_IDS:
    full_result_df[formula_id] = result_dfs[formula_id].to_numpy(copy=False).reshape(-1)
column_order = [
    "runner_date",
    "InnerCode",
    *FORMULA_IDS,
]
full_result_df = full_result_df[column_order]

if OUTPUT_CSV is not None:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    full_result_df.to_csv(OUTPUT_CSV, index=False)
    print("saved:", OUTPUT_CSV)

print("full_result_df shape:", full_result_df.shape)
full_result_df


## 18. 本 notebook 展示了哪些项目能力

| 能力 | 本例中的落点 |
|---|---|
| 统一 FormulaBatch | 5 个输出一次编译、共享 DAG |
| common inputs | `minute_ret → abnormal_volume → pressure_1m → pressure_5m` 只定义一次 |
| CSE / Term 复用 | `pressure_5m` 的 resample Term 被多个下游统计共同引用 |
| 高频真实数据读取 | `stk.1min.close_price` + `stk.1min.turnover_volume` |
| 无 Store 正式数据链路 | 每个任务独立使用 `SmartQuantDataProvider` |
| 同 step 跨日统计 | `intraday_by_step_mean(..., window_days=20)` |
| 日期历史与日内 step 同时存在 | `delay(axis=0)` 与 `step_delay()` 各自作用于不同轴 |
| 显式频率转换 | `resample(1min → 5min, sum)` |
| intraday → daily lowering | `step_sum/std/kurtosis` 把 `5min × 48` 降为 `1d × 1` |
| 日频 rolling | `ts_mean(5/20)`、`ts_std(20)` |
| 自动 lookback | Compiler 沿共享 DAG 推导 `job_lookback` |
| 分块流式执行 | `compute()` 内部使用 `ResultStream` + `chunk_size` |
| workspace 生命周期 | `peak_workspace_values`、`released_terms` |
| 分步性能诊断 | compile、load_many、每个 operator 的耗时与 RSS 曲线 |
| 多输出结果装配 | `ComputeResult.arrays` 返回 5 个完整日频数组 |

这个 example 的重点是：**用户只描述量价逻辑，Compiler 负责把不同时间轴、共享中间结果、lookback 和物理分块组织成一张可执行 DAG。**


## 调用链回顾

```text
FormulaBatch.from_text()
  → bind common inputs + 各输出公式
  → Source describe
  → Domain lowering
       1min × 237
         ↓ resample
       5min × 48
         ↓ step reduction
       1d × 1
  → CSE / reference count
  → 自动 lookback
  → LogicalPlan
  → PhysicalPlanner 按日期分块并扩展 read_dates
  → DataProvider.bind_many()/load_many()
  → Runtime 拓扑执行，共享中间 Term
  → BatchFactorEngine.compute() 内部消费 ResultStream
  → ComputeResult.arrays 装配 5 个完整结果
  → 转为 result_dfs / coverage_df
  → full_result_df + CSV
```
